In [1]:
import hashlib, json, os, shutil, sys, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# Optional: authenticate HF_TOKEN if attached in Secrets
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
except Exception:
    pass

# 1. Locate attached backup archive & extract
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/audit_workspace")
WORK_DIR.mkdir(parents=True, exist_ok=True)

zip_path = next(INPUT_ROOT.rglob("kaggle_working_full_backup.zip"))
print(f"[INPUT] Found backup archive: {zip_path} ({zip_path.stat().st_size / (1024**2):.2f} MB)")

EXTRACT_DIR = WORK_DIR / "extracted"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(EXTRACT_DIR)

HARNESS_DIR = next(EXTRACT_DIR.rglob("kv_spa_harness")).parent
RUN_DIR = next(EXTRACT_DIR.rglob("ten_model_production_run"))
print(f"[PATHS] HARNESS_DIR={HARNESS_DIR} | RUN_DIR={RUN_DIR}")
sys.path.insert(0, str(HARNESS_DIR))

from kv_spa_harness.dataset import PERSONA_REGISTRY
from kv_spa_harness.metrics_and_stats import (
    LocalSemanticPreservationEvaluator,
    compute_benjamini_hochberg_fdr,
    compute_cohens_d_with_bootstrap_ci,
    compute_mattr,
)

# ==============================================================================
# STAGE 1: EXECUTION AUDIT (240 Trajectories x 50 Turns = 12,000 Turns)
# ==============================================================================
json_files = sorted(
    p for p in RUN_DIR.rglob("*.json")
    if p.stem.startswith(("A1_", "A2_", "A3_", "A4_", "A5_", "A6_"))
)
records = [json.loads(p.read_text(encoding="utf-8")) for p in json_files]
incomplete = [r["trajectory_id"] for r in records if len(r.get("turn_records", [])) != 50]

empty_details = [
    (r["ollama_model_id"], r["arm_id"], r["persona_id"], t["turn_index"])
    for r in records
    for t in r.get("turn_records", [])
    if not str(t.get("raw_model_response", "")).strip()
]
manifest_sha = hashlib.sha256("".join(sorted(p.name for p in json_files)).encode()).hexdigest()[:16]

assert len(json_files) == 240 and len(incomplete) == 0, f"Incomplete trajectories: {incomplete}"
print(f"[STAGE 1 PASS] 240/240 trajectories complete (12,000 turns) | Manifest SHA256[:16]: {manifest_sha}")
print(f"[STAGE 1 TELEMETRY] Empty responses: {len(empty_details)} / 12,000 turns ({len(empty_details)/120:.2f}%)")
if empty_details:
    emp_df = pd.DataFrame(empty_details, columns=["model_id", "arm_id", "persona_id", "turn_index"])
    print("  Empty breakdown by model & arm:", emp_df.groupby(["model_id", "arm_id"]).size().to_dict())

# ==============================================================================
# STAGE 2: IMPLEMENTATION AUDIT (A1-A6 Harness Code & Prompt/Hook Signatures)
# ==============================================================================
harness_modules = sorted(str(p.relative_to(HARNESS_DIR)) for p in HARNESS_DIR.rglob("*.py"))
nb_matches = list(INPUT_ROOT.rglob("*.ipynb"))
print(f"\n[STAGE 2 PASS] Attached notebooks: {[p.name for p in nb_matches]}")
print(f"[STAGE 2 PASS] Harness modules ({len(harness_modules)}): {harness_modules}")

arm_token_stats = {
    arm: round(float(np.mean([
        t["input_prompt_token_count"]
        for r in records if r["arm_id"] == arm
        for t in r["turn_records"]
    ])), 1)
    for arm in ["A1", "A2", "A3", "A4", "A5", "A6"]
}
print(f"[STAGE 2 PASS] Mean input prompt tokens per arm: {arm_token_stats}")

# ==============================================================================
# STAGE 3: EVALUATION METRIC RE-COMPUTATION ON RAW RESPONSES
# ==============================================================================
evaluator = LocalSemanticPreservationEvaluator()
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(records), size=10, replace=False)

for idx in sample_indices:
    r = records[int(idx)]
    persona = PERSONA_REGISTRY[r["persona_id"]]
    for t_idx in [0, 14, 34, 49]:
        t = r["turn_records"][t_idx]
        resp = t["raw_model_response"]
        p_sem = evaluator.compute_p_sem(resp, persona)
        mattr = compute_mattr(resp, window_size=50)
        assert 0.0 <= p_sem <= 1.0 and 0.0 <= mattr <= 1.0
print("\n[STAGE 3 PASS] Re-evaluated P_sem & MATTR-50 on 40 sampled raw turns (T1, T15, T35, T50) — all valid.")

# ==============================================================================
# STAGE 4: STATISTICAL DESIGN & CONTRASTS RE-VERIFICATION (N = 40 CELLS/ARM)
# ==============================================================================
traj_rows = []
for r in records:
    persona = PERSONA_REGISTRY[r["persona_id"]]
    p_vals = [evaluator.compute_p_sem(t["raw_model_response"], persona) for t in r["turn_records"]]
    traj_rows.append({
        "arm_id": r["arm_id"],
        "cell_id": f"{r['ollama_model_id']}__{r['persona_id']}",
        "mean_p_sem": float(np.mean(p_vals)),
    })
audit_traj_df = pd.DataFrame(traj_rows)
ref = audit_traj_df[audit_traj_df["arm_id"] == "A2"].set_index("cell_id")["mean_p_sem"]

raw_p, contrast_out = [], []
for arm in ["A1", "A3", "A4", "A5", "A6"]:
    tst = audit_traj_df[audit_traj_df["arm_id"] == arm].set_index("cell_id")["mean_p_sem"].loc[ref.index]
    t_stat, p_val = stats.ttest_rel(tst.values, ref.values)
    d_val, ci_l, ci_h = compute_cohens_d_with_bootstrap_ci(tst.values, ref.values, n_bootstrap=1000)
    raw_p.append(float(p_val))
    contrast_out.append((arm, float(np.mean(tst - ref)), float(t_stat), float(d_val)))

_, fdr_p = compute_benjamini_hochberg_fdr(raw_p)
print("\n[STAGE 4 PASS] Independent Paired Contrasts vs. A2 (N=40 matched cells):")
for (arm, diff, t_s, d_v), p_adj in zip(contrast_out, fdr_p):
    print(f"  {arm} vs A2: mean_diff={diff:+.4f}, t={t_s:.3f}, Cohen_d={d_v:+.3f}, p_fdr={p_adj:.5f}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[INPUT] Found backup archive: /kaggle/input/notebooks/nabhmehta/kv-vs-spa/kaggle_working_full_backup.zip (18.24 MB)
[PATHS] HARNESS_DIR=/kaggle/working/audit_workspace/extracted/kv_vs_spa_exp | RUN_DIR=/kaggle/working/audit_workspace/extracted/kv_vs_spa_exp/outputs/ten_model_production_run
[STAGE 1 PASS] 240/240 trajectories complete (12,000 turns) | Manifest SHA256[:16]: de44ecabd467bc12
[STAGE 1 TELEMETRY] Empty responses: 23 / 12,000 turns (0.19%)
  Empty breakdown by model & arm: {('phi3:medium', 'A1'): 6, ('phi3:medium', 'A3'): 9, ('phi3:medium', 'A4'): 4, ('phi3:medium', 'A5'): 1, ('phi3:medium', 'A6'): 1, ('phi3:mini', 'A3'): 1, ('phi3:mini', 'A4'): 1}

[STAGE 2 PASS] Attached notebooks: ['__notebook__.ipynb', '__notebook__.ipynb']
[STAGE 2 PASS] Harness modules (11): ['kv_spa_harness/__init__.py', 'kv_spa_harness/baselines.py', 'kv_spa_harness/belitsky_steering.py', 'kv_spa_harness/dataset.py', 'kv_spa_harness/foundation.py', 'kv_spa_harness/gcad_and_controls.py', 'kv_spa_harne

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[STAGE 3 PASS] Re-evaluated P_sem & MATTR-50 on 40 sampled raw turns (T1, T15, T35, T50) — all valid.

[STAGE 4 PASS] Independent Paired Contrasts vs. A2 (N=40 matched cells):
  A1 vs A2: mean_diff=-0.0360, t=-6.058, Cohen_d=-0.544, p_fdr=0.00000
  A3 vs A2: mean_diff=-0.0192, t=-4.903, Cohen_d=-0.321, p_fdr=0.00002
  A4 vs A2: mean_diff=-0.0345, t=-6.101, Cohen_d=-0.531, p_fdr=0.00000
  A5 vs A2: mean_diff=-0.0118, t=-1.329, Cohen_d=-0.173, p_fdr=0.19151
  A6 vs A2: mean_diff=-0.0361, t=-6.375, Cohen_d=-0.557, p_fdr=0.00000


In [2]:
# ==============================================================================
# COMPLETE END-TO-END 4-STAGE CRYPTOGRAPHIC, BITWISE & STATISTICAL AUDIT SCRIPT
# ==============================================================================
import ast
import hashlib
import json
import os
import shutil
import sys
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# Optional: authenticate HF_TOKEN if attached in Secrets
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
except Exception:
    pass

# 1. Locate attached backup archive & extract
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/audit_workspace")
WORK_DIR.mkdir(parents=True, exist_ok=True)

zip_path = next(INPUT_ROOT.rglob("kaggle_working_full_backup.zip"))
print(f"[INPUT] Found backup archive: {zip_path} ({zip_path.stat().st_size / (1024**2):.2f} MB)")

EXTRACT_DIR = WORK_DIR / "extracted"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(EXTRACT_DIR)

HARNESS_DIR = next(EXTRACT_DIR.rglob("kv_spa_harness")).parent
RUN_DIR = next(EXTRACT_DIR.rglob("ten_model_production_run"))
print(f"[PATHS] HARNESS_DIR={HARNESS_DIR} | RUN_DIR={RUN_DIR}")
sys.path.insert(0, str(HARNESS_DIR))

from kv_spa_harness.dataset import PERSONA_REGISTRY
from kv_spa_harness.metrics_and_stats import (
    LocalSemanticPreservationEvaluator,
    compute_benjamini_hochberg_fdr,
    compute_cohens_d_with_bootstrap_ci,
    compute_mattr,
    compute_target_marker_density,
    count_hard_constraint_violations,
)

# ==============================================================================
# STAGE 1: FULL SHA-256 ARCHIVE & CORPUS DIGESTS + EMPTY-RESPONSE AUDIT
# ==============================================================================
print("\n" + "=" * 110)
print("STAGE 1: FULL SHA-256 ARCHIVE & CORPUS DIGESTS + EMPTY-RESPONSE AUDIT")
print("=" * 110)
json_files = sorted(
    p for p in RUN_DIR.rglob("*.json")
    if p.stem.startswith(("A1_", "A2_", "A3_", "A4_", "A5_", "A6_"))
)
records = [json.loads(p.read_text(encoding="utf-8")) for p in json_files]
incomplete = [r["trajectory_id"] for r in records if len(r.get("turn_records", [])) != 50]
assert len(json_files) == 240 and len(incomplete) == 0, f"Incomplete trajectories: {incomplete}"

archive_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
corpus_hasher = hashlib.sha256()
for jf in sorted(json_files, key=lambda p: p.name):
    corpus_hasher.update(jf.read_bytes())
corpus_sha256 = corpus_hasher.hexdigest()

print(f"  Archive Size (MB):      {zip_path.stat().st_size / (1024**2):.2f}")
print(f"  Archive Byte SHA-256:   {archive_sha256}")
print(f"  Corpus Content SHA-256: {corpus_sha256}")
print(f"  Completed Trajectories: {len(records)} / 240 (12,000 turns)")

empty_rows = []
for r in records:
    for t in r.get("turn_records", []):
        if not str(t.get("raw_model_response", "")).strip():
            empty_rows.append({
                "trajectory_id": r["trajectory_id"],
                "model_id": r["ollama_model_id"],
                "arm_id": r["arm_id"],
                "persona_id": r["persona_id"],
                "turn_index": int(t["turn_index"]),
            })

emp_full_df = pd.DataFrame(empty_rows)
print(f"  Total zero-token turns: {len(emp_full_df)} / 12000 ({len(emp_full_df)/120.0:.2f}%)")
print(f"  Affected trajectories:  {emp_full_df['trajectory_id'].nunique()}")

emp_pivot = pd.pivot_table(
    emp_full_df,
    index="model_id",
    columns="arm_id",
    values="turn_index",
    aggfunc="count",
    fill_value=0,
).reindex(columns=["A1", "A2", "A3", "A4", "A5", "A6"], fill_value=0)
emp_pivot["Model_Total"] = emp_pivot.sum(axis=1)
emp_pivot["Model_Pct_1200"] = (emp_pivot["Model_Total"] / 1200.0 * 100.0).round(2)
print("\n  TABLE 11 (EMPTY RESPONSES BY MODEL x ARM):")
print(emp_pivot.to_string())
print("  Arm Totals:", emp_pivot[["A1", "A2", "A3", "A4", "A5", "A6"]].sum().to_dict())

boundary_trajs = emp_full_df[
    (emp_full_df["turn_index"] <= 5) | (emp_full_df["turn_index"] >= 46)
]["trajectory_id"].unique()
print(f"  Boundary-window affected trajectories ([1..5] or [46..50]): {len(boundary_trajs)} -> {list(boundary_trajs)}")

# ==============================================================================
# STAGE 2: AST SOURCE AUDIT & TURN-1 / TURN-50 PROMPT TOKEN SIGNATURES
# ==============================================================================
print("\n" + "=" * 110)
print("STAGE 2: AST SOURCE AUDIT & TURN-1 / TURN-50 PROMPT TOKEN SIGNATURES")
print("=" * 110)
target_files = [
    "baselines.py",
    "belitsky_steering.py",
    "sun_transplantation.py",
    "memory_inception.py",
    "runner.py",
    "metrics_and_stats.py",
]
for py_path in sorted(HARNESS_DIR.rglob("*.py")):
    if py_path.name in target_files:
        src = py_path.read_text(encoding="utf-8")
        mod_sha = hashlib.sha256(src.encode("utf-8")).hexdigest()[:16]
        tree = ast.parse(src)
        fns = [n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
        print(f"  {py_path.name:<24} | SHA256[:16]={mod_sha} | funcs={fns}")

for t_step in [1, 50]:
    step_tokens = {
        arm: round(float(np.mean([
            t["input_prompt_token_count"]
            for r in records if r["arm_id"] == arm
            for t in r["turn_records"] if int(t["turn_index"]) == t_step
        ])), 1)
        for arm in ["A1", "A2", "A3", "A4", "A5", "A6"]
    }
    print(f"  Turn {t_step:<2} mean input tokens: {step_tokens}")

# ==============================================================================
# STAGE 3: FULL 12,000-TURN BITWISE METRIC REPRODUCTION VS ARCHIVED CSV
# ==============================================================================
print("\n" + "=" * 110)
print("STAGE 3: FULL 12,000-TURN BITWISE METRIC REPRODUCTION VS ARCHIVED CSV")
print("=" * 110)
turn_csv_path = next(INPUT_ROOT.rglob("full_12000_turn_level_telemetry.csv"))
traj_csv_path = next(INPUT_ROOT.rglob("full_240_trajectory_summary_with_models.csv"))
print(f"  Attached turn CSV: {turn_csv_path}")
print(f"  Attached traj CSV: {traj_csv_path}")

archived_turns_df = pd.read_csv(turn_csv_path)
archived_traj_df = pd.read_csv(traj_csv_path)
evaluator = LocalSemanticPreservationEvaluator()

recomp_rows = []
for r in records:
    persona = PERSONA_REGISTRY[r["persona_id"]]
    for t in r["turn_records"]:
        resp = t["raw_model_response"]
        recomp_rows.append({
            "trajectory_id": r["trajectory_id"],
            "arm_id": r["arm_id"],
            "cell_id": f"{r['ollama_model_id']}__{r['persona_id']}",
            "turn_index": int(t["turn_index"]),
            "p_sem_re": float(evaluator.compute_p_sem(resp, persona)),
            "mattr_re": float(compute_mattr(resp, window_size=50)),
            "dens_re":  float(compute_target_marker_density(resp, persona.diagnostic_lexical_markers)),
            "viol_re":  int(count_hard_constraint_violations(resp, persona.forbidden_tokens_or_phrases)),
        })

recomp_df = pd.DataFrame(recomp_rows)
merged = recomp_df.merge(archived_turns_df, on=["trajectory_id", "turn_index"])
assert len(merged) == 12000, f"Expected 12,000 merged rows, got {len(merged)}"

max_d_psem  = float(np.max(np.abs(merged["p_sem_re"] - merged["p_sem"])))
max_d_mattr = float(np.max(np.abs(merged["mattr_re"] - merged["mattr_50"])))
max_d_dens  = float(np.max(np.abs(merged["dens_re"]  - merged["marker_density"])))
viol_match  = bool((merged["viol_re"] == merged["hard_violations"]).all())

print(f"  Merged rows verified:     {len(merged)} / 12000")
print(f"  max |delta P_sem|:        {max_d_psem:.3e}")
print(f"  max |delta MATTR|:        {max_d_mattr:.3e}")
print(f"  max |delta Marker Dens.|: {max_d_dens:.3e}")
print(f"  violations_match:         {viol_match}")

# ==============================================================================
# STAGE 4: INFERENTIAL CONTRASTS & BOUNDARY-WINDOW SENSITIVITY VERIFICATION
# ==============================================================================
print("\n" + "=" * 110)
print("STAGE 4: PAIRED CONTRASTS (N=40 CELLS) & BOUNDARY-WINDOW SENSITIVITY CHECK")
print("=" * 110)
audit_traj_df = (
    recomp_df.groupby(["arm_id", "cell_id"], as_index=False)["p_sem_re"]
    .mean()
    .rename(columns={"p_sem_re": "mean_p_sem"})
)
ref = audit_traj_df[audit_traj_df["arm_id"] == "A2"].set_index("cell_id")["mean_p_sem"]

raw_p, contrast_out = [], []
for arm in ["A1", "A3", "A4", "A5", "A6"]:
    tst = audit_traj_df[audit_traj_df["arm_id"] == arm].set_index("cell_id")["mean_p_sem"].loc[ref.index]
    t_stat, p_val = stats.ttest_rel(tst.values, ref.values)
    d_val, ci_l, ci_h = compute_cohens_d_with_bootstrap_ci(tst.values, ref.values, n_bootstrap=1000)
    raw_p.append(float(p_val))
    contrast_out.append((arm, float(np.mean(tst - ref)), float(t_stat), float(d_val)))

_, fdr_p = compute_benjamini_hochberg_fdr(raw_p)
print("  Independent Paired Contrasts vs. A2 (N=40 matched cells):")
for (arm, diff, t_s, d_v), p_adj in zip(contrast_out, fdr_p):
    print(f"    {arm} vs A2: mean_diff={diff:+.4f}, t={t_s:.3f}, Cohen_d={d_v:+.3f}, p_fdr={p_adj:.5f}")

no_boundary_df = archived_traj_df[~archived_traj_df["trajectory_id"].isin(boundary_trajs)]
sens_compare = pd.DataFrame({
    "Full_240_Pct": (archived_traj_df.groupby("arm_id")["repetition_collapse_flag"].mean() * 100.0).round(2),
    "Excl_5_Boundary_Pct": (no_boundary_df.groupby("arm_id")["repetition_collapse_flag"].mean() * 100.0).round(2),
})
print("\n  MATTR-Decline Sensitivity Check (Full N=240 vs. Excluding 5 Boundary Trajectories):")
print(sens_compare.to_string())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[INPUT] Found backup archive: /kaggle/input/notebooks/nabhmehta/kv-vs-spa/kaggle_working_full_backup.zip (18.24 MB)
[PATHS] HARNESS_DIR=/kaggle/working/audit_workspace/extracted/kv_vs_spa_exp | RUN_DIR=/kaggle/working/audit_workspace/extracted/kv_vs_spa_exp/outputs/ten_model_production_run

STAGE 1: FULL SHA-256 ARCHIVE & CORPUS DIGESTS + EMPTY-RESPONSE AUDIT
  Archive Size (MB):      18.24
  Archive Byte SHA-256:   68caad94812bcf582ce7ff40c9d0ecadc6f13415df380684f43fa8ad7f91bd14
  Corpus Content SHA-256: d65da777d58a78b7bd693b4d7fb43eb888f1615a1ef7028a2a0bf52ab06a5380
  Completed Trajectories: 240 / 240 (12,000 turns)
  Total zero-token turns: 23 / 12000 (0.19%)
  Affected trajectories:  11

  TABLE 11 (EMPTY RESPONSES BY MODEL x ARM):
arm_id       A1  A2  A3  A4  A5  A6  Model_Total  Model_Pct_1200
model_id                                                        
phi3:medium   6   0   9   4   1   1           21            1.75
phi3:mini     0   0   1   1   0   0            2          

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Merged rows verified:     12000 / 12000
  max |delta P_sem|:        5.551e-17
  max |delta MATTR|:        1.110e-16
  max |delta Marker Dens.|: 3.553e-15
  violations_match:         True

STAGE 4: PAIRED CONTRASTS (N=40 CELLS) & BOUNDARY-WINDOW SENSITIVITY CHECK
  Independent Paired Contrasts vs. A2 (N=40 matched cells):
    A1 vs A2: mean_diff=-0.0360, t=-6.058, Cohen_d=-0.544, p_fdr=0.00000
    A3 vs A2: mean_diff=-0.0192, t=-4.903, Cohen_d=-0.321, p_fdr=0.00002
    A4 vs A2: mean_diff=-0.0345, t=-6.101, Cohen_d=-0.531, p_fdr=0.00000
    A5 vs A2: mean_diff=-0.0118, t=-1.329, Cohen_d=-0.173, p_fdr=0.19151
    A6 vs A2: mean_diff=-0.0361, t=-6.375, Cohen_d=-0.557, p_fdr=0.00000

  MATTR-Decline Sensitivity Check (Full N=240 vs. Excluding 5 Boundary Trajectories):
        Full_240_Pct  Excl_5_Boundary_Pct
arm_id                                   
A1              15.0                12.82
A2              27.5                27.50
A3               7.5                 7.89
A4           

In [3]:
import json
from pathlib import Path
from huggingface_hub import HfApi

# 1. Check exact checkpoints recorded in your 240 trajectory JSONs & manifests
run_dir = Path("/kaggle/working/audit_workspace/extracted/kv_vs_spa_exp/outputs/ten_model_production_run")
seen_models = {}
for jf in sorted(run_dir.glob("A*_*.json")):
    d = json.loads(jf.read_text(encoding="utf-8"))
    mid = d["ollama_model_id"]
    if mid not in seen_models:
        seen_models[mid] = d["hf_checkpoint"]

print("=== 1. ARCHIVED TRAJECTORY JSON CHECKPOINTS ===")
for mid, repo in sorted(seen_models.items()):
    print(f"  {mid:<26} -> {repo}")

# 2. Check pinned revisions in foundation.py & live Hugging Face Hub commit SHAs
from kv_spa_harness.foundation import OFFICIAL_HF_MODEL_SLUGS, DEFAULT_LOCAL_EMBEDDING_CHECKPOINT, DEFAULT_LOCAL_EMBEDDING_REVISION

api = HfApi()
print("\n=== 2. FOUNDATION PINNED REVISIONS VS LIVE HF HUB ===")
emb_info = api.model_info(DEFAULT_LOCAL_EMBEDDING_CHECKPOINT, revision=DEFAULT_LOCAL_EMBEDDING_REVISION)
print(f"  {'Embedding Evaluator':<26} | {DEFAULT_LOCAL_EMBEDDING_CHECKPOINT:<38} | pinned={DEFAULT_LOCAL_EMBEDDING_REVISION} | valid={emb_info.sha == DEFAULT_LOCAL_EMBEDDING_REVISION}")

for mid, mapping in OFFICIAL_HF_MODEL_SLUGS.items():
    repo = seen_models.get(mid, mapping.hf_instruct_slug)
    rev = "main" if repo != mapping.hf_instruct_slug else mapping.default_revision
    try:
        info = api.model_info(repo, revision=rev)
        print(f"  {mid:<26} | {repo:<38} | used={rev:<40} | resolved_sha={info.sha}")
    except Exception as e:
        print(f"  {mid:<26} | {repo:<38} | used={rev:<40} | hub_check={type(e).__name__}")

=== 1. ARCHIVED TRAJECTORY JSON CHECKPOINTS ===
  gemma2:9b                  -> google/gemma-2-9b-it
  llama3.1:8b                -> meta-llama/Llama-3.1-8B-Instruct
  llama3:8b                  -> NousResearch/Meta-Llama-3-8B-Instruct
  llama3:8b-instruct-q4_K_M  -> NousResearch/Meta-Llama-3-8B-Instruct
  mistral-nemo:latest        -> mistralai/Mistral-Nemo-Instruct-2407
  mistral:7b                 -> mistralai/Mistral-7B-Instruct-v0.3
  phi3:medium                -> microsoft/Phi-3-medium-4k-instruct
  phi3:mini                  -> microsoft/Phi-3-mini-4k-instruct
  qwen2.5:3b                 -> Qwen/Qwen2.5-3B-Instruct
  qwen2.5:7b                 -> Qwen/Qwen2.5-7B-Instruct

=== 2. FOUNDATION PINNED REVISIONS VS LIVE HF HUB ===
  Embedding Evaluator        | BAAI/bge-small-en-v1.5                 | pinned=5c38ec7c405ec4b44b94cc5a9bb96e735b38267a | valid=True
  llama3.1:8b                | meta-llama/Llama-3.1-8B-Instruct       | used=0e9e39f249a16976918f6564b8830bc894c89659 | reso

In [4]:
import ast
import hashlib
from pathlib import Path

harness_dir = Path("/kaggle/working/audit_workspace/extracted/kv_vs_spa_exp/kv_spa_harness")

expected_table_hashes = {
    "baselines.py": "99cc2662d7db4e78",
    "belitsky_steering.py": "5dfe93067bd8de5c",
    "sun_transplantation.py": "0f45e10a5c684403",
    "memory_inception.py": "783be8470e9ffd12",
    "metrics_and_stats.py": "a61316c59257139d",
    "runner.py": "db6575e174af0dba",
}

print(f"{'Module Path':<38} | {'SHA-256[:16]':<16} | {'Table Match':<11} | {'AST Classes & Functions'}")
print("-" * 120)

for py_file in sorted(harness_dir.glob("*.py")):
    if py_file.name == "__init__.py":
        continue
    src = py_file.read_text(encoding="utf-8")
    digest16 = hashlib.sha256(src.encode("utf-8")).hexdigest()[:16]
    tree = ast.parse(src)
    defs = [
        node.name
        for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.ClassDef))
    ]
    exp = expected_table_hashes.get(py_file.name)
    status = ("EXACT" if digest16 == exp else f"MISMATCH({exp})") if exp else "EXTRA_MOD"
    print(f"kv_spa_harness/{py_file.name:<23} | {digest16:<16} | {status:<11} | {', '.join(defs)}")

Module Path                            | SHA-256[:16]     | Table Match | AST Classes & Functions
------------------------------------------------------------------------------------------------------------------------
kv_spa_harness/baselines.py            | 99cc2662d7db4e78 | EXACT       | TurnExecutionTelemetry, TrajectoryRecord, build_turn_messages, execute_single_turn_generation, run_baseline_trajectory
kv_spa_harness/belitsky_steering.py    | 5dfe93067bd8de5c | EXACT       | ContrastivePromptPair, BelitskySteeringConfig, PersonaSteeringVectors, resolve_target_layers, extract_contrastive_steering_vectors, apply_belitsky_kv_steering, execute_belitsky_steered_turn_generation, run_belitsky_steering_trajectory
kv_spa_harness/dataset.py              | b04a32e20e8c72e5 | EXTRA_MOD   | PersonaRubricCriteria, PersonaProfile, ThematicBlockID, DialogueTurnPrompt, generate_standardized_50_turn_dataset, ExportedDatasetManifest, export_dialogue_dataset
kv_spa_harness/foundation.py           | 